In [10]:
# setup connecting to llm via api key
from dotenv import load_dotenv
import os
load_dotenv()


from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [11]:
# function to talk to llm
MODEL = "openai/gpt-oss-20b"

def llm(prompt, model=MODEL):
    response = openai_client.responses.create(
        model=model,
        input=prompt
    )
    return response.output_text

llm("Hey! how are you?")

'Hey! I’m doing great—thanks for asking. How about you? Anything interesting happening on your end?'

In [12]:
# list models available to your Groq API key
models = openai_client.models.list()
sorted([m.id for m in models.data])[:30]

['allam-2-7b',
 'canopylabs/orpheus-arabic-saudi',
 'canopylabs/orpheus-v1-english',
 'groq/compound',
 'groq/compound-mini',
 'llama-3.1-8b-instant',
 'llama-3.3-70b-versatile',
 'meta-llama/llama-4-scout-17b-16e-instruct',
 'meta-llama/llama-prompt-guard-2-22m',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-120b',
 'openai/gpt-oss-20b',
 'openai/gpt-oss-safeguard-20b',
 'qwen/qwen3-32b',
 'whisper-large-v3',
 'whisper-large-v3-turbo']

In [13]:
## Asking the model coourse specifific question
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Sure thing!  
Just let me know the course name (or ID) and whether you’re already registered in the system. Once I have that info, I’ll check the current capacity and let you know if there’s room for you or if you’d need to join a wait‑list. If you’d like, I can also walk you through the enrollment steps right now.


In [14]:
## Adding context in other to give more streamlines answer...
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [15]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [16]:
# Instead of sending the raw questions directly .. we wuld sned the prompt whcih contain both questions and context.
answer = llm(prompt)
print(answer)

Yes, you can still join, but if you want to receive a certificate you’ll need to submit your project while we’re still accepting submissions.


In [17]:
# RAG stands for Retrieval-Augmented Generation. The two key words are generation and retrieval: generation is the LLM producing text, and retrieval is search. We use search to augment what the LLM generates.

# We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates.

# Search gives the LLM the context it needs to answer correctly.

# Right now we used a naive way of selecting context - we knew in advance which FAQ entry contained the answer. But what we want is to perform search automatically - find the most relevant documents and send those to the LLM.
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

In [18]:
## Fetching COurse FAQ dATASET

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [19]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [20]:
# # SEARCH
# At its core, every search engine does the same thing. It takes a query, scores every document for similarity, and returns the top results.

In [21]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [22]:
# So the above creates an in meemory datasctructures that allow for easy searching later... it tokenizes the text files and filters exactly based on the keyword strings

In [23]:
# TRYING A SEARCH USING THE INDEX BUILT.. EARLIER
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "mlops-zoomcamp"},
    num_results=5
)

search_results


[{'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "O

In [24]:
# wrapping everything up in a search function
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [25]:
search_results = search(question)
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [26]:
# LESSON 6 --> Building the Prompt
# Typically when we build AI systems, the prompt consists of two parts:

#     Instructions (also called system prompt): this tells the LLM how to behave. It never changes - it's the same for every request.
#     User prompt: this changes with every request. It contains the actual question and the retrieved context.


In [27]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [28]:
# Building COntext . The Context is a formatted string with all the search results..

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [29]:
# Building the prompt. Here we will combine the question with the user prompt.

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


In [30]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [31]:
# The prompt is the bridge between search and the LLM. A bad prompt lets the LLM ignore the context and hallucinate. A good prompt keeps the answer grounded.

# Prompt engineering is part art, part science. You experiment, try different things, and see what works. Later in the course we cover evaluation metrics so you can measure how well your prompt performs instead of guessing. For now, this template is a good starting point.

In [32]:
# LESSON 7 Sending Prompt to the LLM

response = openai_client.chat.completions.create(
    model=MODEL,
    messages = [{"role": "user", "content": prompt}]
)

In [33]:
response.choices[0].message.content

'Yes, you can join.  \nYou’re allowed to enroll at any time, but to earn a certificate you must finish the course while it’s still running—that means submitting your Capstone project before the registration period closes. Once the form is closed and the peer‑review list is compiled, the course is considered finished and you’ll no longer receive a certificate.'

In [34]:
response.usage

CompletionUsage(completion_tokens=129, prompt_tokens=399, total_tokens=528, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=48, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.178679607, prompt_time=0.023635276, completion_time=0.153177472, total_time=0.176812748)

In [35]:
#Calculating the Price.. of chatcompletions usage..
# this inpu price is based on this model.. "gpt-5.4-mini:" although this  is not the model that we're currently using
completion_price = 0.75 / 1_000_000
prompt_price = 4.50 / 1_000_000

cost = (
    response.usage.completion_tokens * completion_price+
    response.usage.prompt_tokens * prompt_price
)

cost

0.00189225

In [36]:
# # MESSAGE HISTORY
# Previously we sent only one string as input and got back a response. In practice, you typically send a message history - a list of messages where each message has a role.

# Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.

In [37]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message_history
)

# This separates the fixed instructions from the user prompt, which changes every request.

# OpenAI accepts both developer and system for the instruction role. There may be some difference between them, but in practice I don't see it change the result either way. We use developer in this course.

In [38]:
# THE LLM fUNCTION
# We can now put this together into an updated llm function.
# It now takes both instructions and the user prompt:

In [39]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )

    return response.choices[0].message.content

In [40]:
# Full RAG
# With search, the prompt, and the LLM ready, we wire them together:
def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes – you can still join. Just be sure to submit your project while we’re still accepting submissions if you want to earn a certificate.
